# Predicting Corporate Bond Illiquidity Using Machine Learning
## A Quantitative Study of Market, Structural, and Firm Indicators

**Author:** Sakshi Yadav  
**Data:** WRDS — TRACE · FISD · Compustat North America · CRSP · FRED  
**Period:** January 2018 – December 2023  
**Observations:** 88,025 bond-month observations  

---

### Research Questions
1. Do ML ensemble models (Random Forest, XGBoost) outperform logistic regression in predicting corporate bond illiquidity?
2. Which predictor variables — bond structural, market microstructure, or firm fundamentals — matter most?
3. Are model predictions reliable during market stress (COVID-19 shock, 2022 rate hike cycle)?

### Key Finding
Time to maturity and issue size — structural features fixed at bond issuance — outperformed bid-ask spread history by **3.5×** as predictors of future illiquidity.

---


## 1. Setup and Configuration

In [ ]:
# === CONFIGURATION ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, roc_curve, precision_recall_curve,
    average_precision_score, confusion_matrix
)
from xgboost import XGBClassifier
import shap

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All libraries loaded successfully.')

## 2. Data Loading and Preprocessing

### Data Sources (via WRDS — Wharton Research Data Services)
- **FISD / TRACE**: Bond-level monthly returns, bid-ask spreads, trading volumes, credit ratings, structural characteristics
- **Compustat North America**: Firm-level financial statements (profitability, leverage, coverage, liquidity ratios)
- **CRSP / FRED**: Market-wide indicators

### Sample Filters
- Fixed-coupon, USD-denominated corporate bonds
- January 2018 – December 2023
- Minimum trading history required
- Both active and defaulted bonds included (no survivorship bias)


In [ ]:
# === LOAD DATA ===
# Replace with your local file path after downloading from WRDS
bond = pd.read_csv('data/bond_data.csv', low_memory=False)
firm = pd.read_csv('data/firm_level_financial.csv', low_memory=False)

print(f'Bond data: {bond.shape[0]:,} rows × {bond.shape[1]} columns')
print(f'Firm data: {firm.shape[0]:,} rows × {firm.shape[1]} columns')
print(f'Bond date range: {bond["DATE"].min()} to {bond["DATE"].max()}')

In [ ]:
# === CLEAN NUMERIC FIELDS ===
# FISD/TRACE data comes with currency and percentage formatting
def clean_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('[$,%]', '', regex=True).str.replace(',', '', regex=True),
        errors='coerce'
    )

bond['DATE']             = pd.to_datetime(bond['DATE'], dayfirst=True)
bond['T_Volume']         = clean_num(bond['T_Volume'])
bond['T_Spread']         = clean_num(bond['T_Spread']) / 100  # convert % to decimal
bond['RET_EOM']          = clean_num(bond['RET_EOM'])  / 100
bond['COUPON']           = clean_num(bond['COUPON'])
bond['AMOUNT_OUTSTANDING'] = clean_num(bond['AMOUNT_OUTSTANDING'])

firm['public_date'] = pd.to_datetime(firm['public_date'], dayfirst=True)

# Filter to 2018–2023
bond = bond[(bond['DATE'] >= '2018-01-01') & (bond['DATE'] <= '2023-12-31')].copy()
print(f'Bond rows after date filter: {len(bond):,}')

## 3. Feature Engineering

In [ ]:
# === LAG FEATURES ===
# All predictors lagged to prevent look-ahead bias
# Models can only use information available at prediction time
bond = bond.sort_values(['CUSIP', 'DATE'])

for lag in [1, 2, 3]:
    bond[f'spread_lag{lag}'] = bond.groupby('CUSIP')['T_Spread'].shift(lag)
    bond[f'ret_lag{lag}']    = bond.groupby('CUSIP')['RET_EOM'].shift(lag)

bond['vol_lag1'] = bond.groupby('CUSIP')['T_Volume'].shift(1)

# Amihud (2002) illiquidity measure: |return| / volume
# Higher = less liquid (bigger price impact per dollar traded)
bond['amihud'] = bond['RET_EOM'].abs() / bond['T_Volume'].replace(0, np.nan)
bond['amihud_lag1'] = bond.groupby('CUSIP')['amihud'].shift(1)

print('Lag features created.')

In [ ]:
# === TARGET VARIABLE: illiquid_next ===
# A bond is classified as illiquid if its bid-ask spread
# exceeds the 90th percentile of the cross-sectional distribution
# in the FOLLOWING month.
# Threshold recalculated monthly — relative, not absolute.

bond['ym'] = bond['DATE'].dt.to_period('M')
p90 = bond.groupby('ym')['T_Spread'].transform(lambda x: x.quantile(0.90))
bond['illiquid']      = (bond['T_Spread'] > p90).astype(float)
bond['illiquid_next'] = bond.groupby('CUSIP')['illiquid'].shift(-1)  # forward-looking target

# Secondary target: credit downgrade within next 12 months
bond['rating_next12']  = bond.groupby('CUSIP')['RATING_NUM'].shift(-12)
bond['downgrade_next'] = (bond['rating_next12'] > bond['RATING_NUM']).astype(float)

print(f'Target positive rate (illiquid_next): {bond["illiquid_next"].mean():.1%}')

In [ ]:
# === FIRM-LEVEL DATA MERGE ===
# Link bond panel to Compustat firm fundamentals via CUSIP
# Using public_date (date financials became publicly available) — prevents look-ahead bias

firm['ym']     = firm['public_date'].dt.to_period('M')
bond['cusip6'] = bond['CUSIP'].astype(str).str[:6]
firm['cusip6'] = firm['cusip'].astype(str).str.strip().str[:6]

FIRM_COLS = ['cusip6','ym','roa','roe','de_ratio','intcov_ratio',
             'curr_ratio','quick_ratio','debt_assets','dltt_be',
             'debt_ebitda','cash_debt','bm','ptb']

firm_sub = firm[FIRM_COLS].dropna(subset=['cusip6'])
panel    = bond.merge(firm_sub, on=['cusip6','ym'], how='left')

print(f'Panel shape: {panel.shape}')
print(f'Firm data match rate: {panel["roa"].notna().mean():.1%}')

In [ ]:
# === FINAL CLEAN PANEL ===
FEATURES = [
    # Market microstructure
    'spread_lag1','spread_lag2','spread_lag3',
    'vol_lag1','ret_lag1','ret_lag2','ret_lag3','amihud_lag1',
    # Bond structural characteristics
    'COUPON','TMT','AMOUNT_OUTSTANDING','RATING_NUM','NCOUPS',
    # Firm-level fundamentals (Compustat)
    'roa','roe','de_ratio','intcov_ratio','curr_ratio',
    'quick_ratio','debt_assets','dltt_be','debt_ebitda','cash_debt','bm','ptb'
]

# Drop rows with missing target or core predictors
panel_clean = panel.dropna(subset=['illiquid_next','spread_lag1','vol_lag1','ret_lag1'])

# Median imputation for missing firm variables
for f in FEATURES:
    if f not in panel_clean.columns:
        panel_clean[f] = 0
    panel_clean[f] = panel_clean[f].fillna(panel_clean[f].median())

print(f'Final panel: {len(panel_clean):,} observations')
print(f'Features: {len(FEATURES)}')
print(f'Illiquid rate: {panel_clean["illiquid_next"].mean():.1%}')

## 4. Train-Test Split

**Critical design choice:** Time-based split, NOT random.

Random cross-validation would cause data leakage — the same bond appearing in both train and test folds in adjacent months. A chronological split simulates real-world forecasting where you can only use past data to predict the future.

- **Train:** All observations before July 2023
- **Test:** July – December 2023 (held-out, never seen during training)


In [ ]:
# === TIME-BASED TRAIN/TEST SPLIT ===
panel_clean['DATE'] = pd.to_datetime(panel_clean['DATE'])

train = panel_clean[panel_clean['DATE'] < '2023-07-01']
test  = panel_clean[panel_clean['DATE'] >= '2023-07-01']

X_train = train[FEATURES]
y_train = train['illiquid_next']
X_test  = test[FEATURES]
y_test  = test['illiquid_next']

# Scale for logistic regression only (tree models are scale-invariant)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {len(train):,} obs | Test: {len(test):,} obs')
print(f'Train illiquid rate: {y_train.mean():.1%} | Test: {y_test.mean():.1%}')
print(f'Class imbalance ratio: {(y_train==0).sum()/(y_train==1).sum():.1f}:1 (liquid:illiquid)')

## 5. Model Training

Three models representing increasing complexity:
1. **Logistic Regression** — linear baseline (Altman 1968 tradition)
2. **Random Forest** — ensemble, handles non-linearities via bootstrap aggregation
3. **XGBoost (Balanced)** — sequential gradient boosting with class-weight adjustment for imbalanced target

Class imbalance (~9:1 liquid:illiquid) handled explicitly in all models.


In [ ]:
# === MODEL 1: LOGISTIC REGRESSION (BASELINE) ===
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
lr.fit(X_train_sc, y_train)
lr_prob = lr.predict_proba(X_test_sc)[:,1]
print(f'Logistic — AUC: {roc_auc_score(y_test, lr_prob):.4f}')

In [ ]:
# === MODEL 2: RANDOM FOREST ===
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:,1]
print(f'Random Forest — AUC: {roc_auc_score(y_test, rf_prob):.4f}')

In [ ]:
# === MODEL 3: XGBOOST (BALANCED) ===
scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()  # ~9.8

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,  # upweights illiquid minority class
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train, y_train)
xgb_prob = xgb.predict_proba(X_test)[:,1]
print(f'XGBoost — AUC: {roc_auc_score(y_test, xgb_prob):.4f}')

## 6. Model Evaluation

In [ ]:
# === FULL RESULTS TABLE ===
def evaluate(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'Model': name,
        'AUC':       round(roc_auc_score(y_true, y_prob), 4),
        'Avg Precision': round(average_precision_score(y_true, y_prob), 4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
    }

results = pd.DataFrame([
    evaluate('Logistic Regression', y_test, lr_prob),
    evaluate('Random Forest',       y_test, rf_prob),
    evaluate('XGBoost (Balanced)',  y_test, xgb_prob),
])
print(results.to_string(index=False))
results.to_csv('outputs/model_comparison.csv', index=False)

In [ ]:
# === ROC CURVES ===
fig, ax = plt.subplots(figsize=(8, 6))
for name, prob, color in [
    ('Logistic Regression', lr_prob,  '#2196F3'),
    ('Random Forest',       rf_prob,  '#4CAF50'),
    ('XGBoost (Balanced)',  xgb_prob, '#FF5722'),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models (Test Set: July–Dec 2023)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/roc_comparison.png', dpi=150)
plt.show()

## 7. SHAP Feature Importance Analysis

In [ ]:
# === GLOBAL SHAP ANALYSIS ===
# SHAP (SHapley Additive exPlanations) — Lundberg & Lee (2017)
# Decomposes each prediction into feature contributions
# Guarantees: local accuracy, consistency, missingness

sample = X_test.sample(min(1000, len(X_test)), random_state=RANDOM_STATE)
explainer  = shap.TreeExplainer(xgb)
shap_vals  = explainer.shap_values(sample)

# Global importance: mean absolute SHAP value per feature
shap_df = pd.DataFrame({
    'feature':        FEATURES,
    'mean_abs_shap':  np.abs(shap_vals).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print('Top 10 features by SHAP importance:')
print(shap_df.head(10).to_string(index=False))
shap_df.to_csv('outputs/shap_importance.csv', index=False)

In [ ]:
# === SHAP SUMMARY PLOT ===
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, sample, feature_names=FEATURES, show=False, max_display=15)
plt.title('SHAP Feature Importance — XGBoost (Top 15)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key finding: TMT (time to maturity) is the dominant predictor by a factor of 3.5x over spread_lag1')

## 8. Stress Period Robustness Analysis

Models are applied to three distinct stress sub-periods **without re-estimation**.

This tests whether a model trained on historical data generalises to different market regimes — the real-world test of practical deployability.

| Period | Market Condition |
|--------|-----------------|
| COVID-19 Shock (Mar–Dec 2020) | Pandemic liquidity crisis — spreads spiked, volumes collapsed |
| Rate Hike Cycle (Mar–Dec 2022) | Fed tightening — duration repricing, spread widening |
| Post-Hike Adjustment (Jan–Jun 2023) | Normalisation with residual uncertainty |

**Note:** These sub-periods are within the training window (2018–2023). Performance reflects in-sample generalisation across market regimes, not true out-of-sample forecasting. Only the July–December 2023 test set is genuinely out-of-sample.


In [ ]:
# === STRESS PERIOD ROBUSTNESS ===
stress_periods = {
    'COVID-19 Shock (2020)':    ('2020-03-01', '2020-12-31'),
    'Rate Hike Cycle (2022)':   ('2022-03-01', '2022-12-31'),
    'Post-Hike (2023 H1)':      ('2023-01-01', '2023-06-30'),
}

robust_rows = []
for period, (start, end) in stress_periods.items():
    sub  = panel_clean[(panel_clean['DATE'] >= start) & (panel_clean['DATE'] <= end)]
    Xs   = sub[FEATURES]
    ys   = sub['illiquid_next']
    for name, model in [('Random Forest', rf), ('XGBoost', xgb)]:
        prob = model.predict_proba(Xs)[:,1]
        pred = (prob >= 0.5).astype(int)
        robust_rows.append({
            'Period': period, 'Model': name, 'N': len(sub),
            'AUC':       round(roc_auc_score(ys, prob), 3),
            'F1':        round(f1_score(ys, pred, zero_division=0), 3),
            'Precision': round(precision_score(ys, pred, zero_division=0), 3),
            'Recall':    round(recall_score(ys, pred, zero_division=0), 3),
        })

robust_df = pd.DataFrame(robust_rows)
print(robust_df.to_string(index=False))
robust_df.to_csv('outputs/robustness_results.csv', index=False)

## 9. Results Summary

### Main Findings

| Model | AUC | F1 | Recall |
|-------|-----|----|--------|
| Logistic Regression | 0.864 | 0.385 | 0.784 |
| Random Forest | **0.881** | 0.392 | **0.844** |
| XGBoost (Balanced) | 0.878 | 0.388 | 0.805 |

### SHAP Feature Importance — Top 5
| Rank | Feature | Mean |SHAP| | Category |
|------|---------|-------------|----------|
| 1 | TMT (Time to Maturity) | 1.336 | Bond structural |
| 2 | AMOUNT_OUTSTANDING (Issue size) | 0.835 | Bond structural |
| 3 | spread_lag1 (Bid-ask spread) | 0.381 | Market microstructure |
| 4 | COUPON | 0.240 | Bond structural |
| 5 | amihud_lag1 | 0.211 | Market microstructure |

**Key insight:** Structural features (TMT, issue size) outperformed market microstructure signals by 3.5×.

### Stress Period Performance (XGBoost)
| Period | AUC | Recall |
|--------|-----|--------|
| COVID-19 Shock (2020) | 0.943 | 96.2% |
| Rate Hike Cycle (2022) | 0.935 | 95.3% |
| Post-Hike (2023 H1) | 0.938 | 94.0% |

---

### References
- Altman, E.I. (1968) *The Journal of Finance*, 23(4), 589–609
- Amihud, Y. (2002) *Journal of Financial Markets*, 5(1), 31–56
- Brunnermeier & Pedersen (2009) *Review of Financial Studies*, 22(6), 2201–2238
- Cabrol, Drobetz, Otto & Puhan (2024) *Financial Analysts Journal*, 80(3), 103–127
- Gu, Kelly & Xiu (2020) *Review of Financial Studies*, 33(5), 2223–2273
- Lessmann et al. (2015) *European Journal of Operational Research*, 247(1), 124–136
- Lundberg & Lee (2017) *NeurIPS*, 30, 4765–4774
- Merton (1974) *The Journal of Finance*, 29(2), 449–470
- Pastor & Stambaugh (2003) *Journal of Political Economy*, 111(3), 642–685
- Roll (1984) *Journal of Finance*, 39(4), 1127–1139
